# The REG103_IPC Table

Welcome to this notebook where we go through table **REG103_IPC** of PATSTAT Register. Each row of this table has a string containing 0, 1 or more IPC symbols of an application as published in the European Patent Bulletin. The individual IPC symbols are separated by semicolons. 

In [1]:
from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import REG103_IPC
from sqlalchemy import func
import pandas as pd

# Initialise the PATSTAT client
patstat = PatstatClient(env='PROD')

# Access ORM
db = patstat.orm()

## ID (Primary Key)

Technical identifier for an application, without business meaning. Its values will not change from one PATSTAT edition to the next.

In [2]:
i = db.query(
    REG103_IPC.id
).limit(1000)

df = patstat.df(i)
df

,id
0,20178163
1,4702285
2,22934705
3,16202705
4,98923273
...,...
995,20188932
996,14731940
997,15810997
998,1102918


## IPC_TEXT

IPC classification of the patent. The text is a string and it can contain up to 500 characters. The default value is an empty string.

The data is not updated on a regular basis. Only due to a request for publication in the European Patent Bulletin, the IPC data is updated. Up-to-date classification information, like reclassified IPCs, can be retrieved by linking the PATSTAT EP Register data with PATSTAT Global table TLS209_APPLN_IPC via the `appln_id` attribute.

In [3]:
ipc_text = db.query(
    REG103_IPC.ipc_text,
    REG103_IPC.id
).filter(
    REG103_IPC.ipc_text != ''
).limit(1000)

ipc_text_df = patstat.df(ipc_text)
ipc_text_df

,ipc_text,id
0,"G01S7/481, G01S7/497, G01S17/931",20178163
1,H01L21/8247,4702285
2,"H05B3/14, H05B3/02",22934705
3,F16D65/54,16202705
4,"B65D85/52, B65B25/02",98923273
...,...,...
995,"A23J1/00, A23J3/14, A23J3/28, A61K8/97, A61K36...",15810997
996,"H01R13/115, H01R43/16, H01R13/187",1102918
997,"F16K11/085, F16K5/04, F16K5/18, F16K11/076",23906546
998,"B60R17/02, F16N7/38, F16N29/02",17181235


We can see which applications are associated with the highest number of symbols.

In [4]:
# Create an empty list to add the number of symbols for each application id
num_labels = []

# Iterate over the list of symbols (over the rows under "ipc_text")
for label in ipc_text_df['ipc_text']:
    # Convert label in a list: a new element is defined every time a comma is encountered
    sequence = label.split(",")
    # Append the length of list just created
    num_labels.append(len(sequence))

# Create a new column of the dataframes containing the length of the symbols sequence
ipc_text_df['number_of_labels'] = num_labels
# Sort the dataframe by the number_of_labels attribute in descending order
ipc_text_df = ipc_text_df.sort_values(by='number_of_labels', ascending=False)
# Save the top 10 applications
most_labels = ipc_text_df.head(10)
most_labels

,ipc_text,id,number_of_labels
67,"C07D471/10, A61K31/445, A61K31/4965, C07D241/5...",16845836,19
995,"A23J1/00, A23J3/14, A23J3/28, A61K8/97, A61K36...",15810997,16
871,"B60W10/02, B60K6/48, B60K6/54, B60L11/14, B60W...",9716867,15
498,"C07D401/14, C07D405/14, C07D215/38, C07D401/12...",22153996,15
460,"C10L1/198, C10L10/08, C10M145/34, C08G65/00, C...",14786833,15
559,"C25D5/02, C23C18/16, C23C18/30, H05K3/00, C23F...",18826345,14
810,"C03B9/16, C03B9/193, C03B9/29, C03B9/34, C03B9...",10178956,14
347,"C08F220/04, C08F216/04, C08F220/20, H01M10/05,...",19858275,14
909,"C22C38/02, C22C38/04, C22C38/22, C22C38/32, C2...",21922314,13
903,"H04W28/08, H04W36/22, H04W36/00, H04W8/26, H04...",10772233,12


As of now, we have the number of labels associated to each application. In the following, we retrieve the number of occurrences of a certain label in the database.

In [5]:
from collections import Counter
import matplotlib.pyplot as plt

# Take the ipc_text column and convert it to a list object
list_of_labels = list(ipc_text_df['ipc_text'].values)

# Count the occurrences of each element
element_counts = Counter(list_of_labels)

# Filter elements with a count greater than or equal to 7
filtered_counts = {k: v for k, v in element_counts.items() if v >= 2}

# Get the elements and their counts sorted by frequency
elements, counts = zip(*sorted(filtered_counts.items(), key=lambda x: x[1], reverse=True))
for element, count in zip(elements, counts):
    print(element, count)


G06F17/30 3
G06F15/16 3
E04D13/18, H02S20/23 2
H04L12/56 2
H01J37/32 2
H01S3/08 2
H04L29/06 2
G06F17/60 2
H04W72/12 2
A61B18/14 2
H04L12/24 2
A61F2/02 2
C09D11/00 2
G06F17/00 2


Finally, we can count the number of empty texts.

In [6]:
no_text = db.query(
    func.count(REG103_IPC.id).label('total')
).filter(
    REG103_IPC.ipc_text == ''
)

no_text_df = patstat.df(no_text)
no_text_df['total'].values.item()

18070

## CHANGE_DATE

It is the date of when the record was saved in the database.

In [7]:
change_date = db.query(
    REG103_IPC.change_date,
    REG103_IPC.id
).limit(100)

change_date_df = patstat.df(change_date)
change_date_df

,change_date,id
0,2000-03-31,99945696
1,2024-04-10,21845876
2,2015-04-21,13781090
3,2001-03-23,420226
4,1999-01-08,98306187
...,...,...
95,2010-06-11,7766563
96,2017-07-05,15719017
97,1993-12-10,92906456
98,2021-09-24,19828529


We can retrieve the applications for which a record was saved in a particular year, let's say 2020.

In [8]:
cd = db.query(
    REG103_IPC.change_date,
    REG103_IPC.id
).filter(
    REG103_IPC.change_date > '2019-12-31',
    REG103_IPC.change_date < '2021-01-01'
)

cd_df = patstat.df(cd)
cd_df

,change_date,id
0,2020-10-12,18807174
1,2020-04-01,17805678
2,2020-04-13,17908739
3,2020-12-21,19741774
4,2020-10-26,18780096
...,...,...
372504,2020-12-25,19743103
372505,2020-12-25,19771525
372506,2020-12-25,15810677
372507,2020-12-25,19711389


## BULLETIN_YEAR

For actions that have been published in the European Patent Bulletin, it is the year of the publication in the European Patent Bulletin. The default value is 0, used for applications that are not published or for which the year is not known. The format is YYYY otherwise.

In [9]:
years = db.query(
    REG103_IPC.bulletin_year,
    REG103_IPC.id
).limit(1000)

years_df = patstat.df(years)
years_df

,bulletin_year,id
0,2020,20178163
1,2005,4702285
2,2025,22934705
3,2018,16202705
4,2000,98923273
...,...,...
995,2021,20188932
996,2016,14731940
997,2018,15810997
998,2002,1102918


## BULLETIN_NR

This is the issue number of the European Patent Bulletin for actions that have been published in it. This attribute indicates the calendar week the European Patent Bulletin was published. The default value 0 is used when the attribute `bulletin_year` is 0.

In [10]:
bulletin_nr = db.query(
    REG103_IPC.id,
    REG103_IPC.bulletin_nr,
    REG103_IPC.bulletin_year
).limit(100)

bulletin_nr_df = patstat.df(bulletin_nr)
bulletin_nr_df

,id,bulletin_nr,bulletin_year
0,98305574,25,2000
1,15192015,19,2016
2,16834035,0,0
3,5784449,0,0
4,24872322,0,0
...,...,...,...
95,20930462,0,0
96,87107374,49,1987
97,21156351,24,2021
98,8167234,49,2013
